# 01 — Data Access Pilot & Verification (Stage 1)

### What this notebook does:
1. **Probes data sources** (verifies that Arctic Shift Reddit API is reachable).
2. **Demonstrates the text cleaner & tokenizer** (shows before/after examples of how URLs, user tags, emojis, and negations like *not* / *never* are preserved).
3. **Pulls a tiny pilot slice** (e.g. 1 sample week of data) to prove data ingestion before running full scale.
4. **Displays an inspection table** so you can visually confirm data quality.

---

In [ ]:
# =============================================================================
# Cell 1 — USER SETTINGS & DECISIONS (EDIT THIS CELL)
# =============================================================================

# 1. CHOOSE YOUR TARGET SCOPE:
#    - "SUBREDDIT_LIST" : Pull data from specific subreddits (e.g., AskAcademia, PhD)
#    - "ALL_REDDIT"     : Pull a sample from the entire platform (all subreddits combined)
PILOT_MODE = "SUBREDDIT_LIST"  # Options: "SUBREDDIT_LIST" or "ALL_REDDIT"

# 2. SUBREDDIT SELECTION (Only used if PILOT_MODE == "SUBREDDIT_LIST"):
#    Enter the subreddit names you want to test:
PILOT_SUBS = ["AskAcademia", "Academia", "PhD"]

# 3. PILOT SAMPLE WEEKS (3 sample weeks across historical eras):
PILOT_WEEKS = [
    ("2015-06-01", "2015-06-08"),   # Historical era
    ("2019-01-01", "2019-01-08"),   # Pre-COVID baseline
    ("2023-07-01", "2023-07-08")    # Recent post-API regime
]

# 4. SAFETY CAP:
MAX_RECORDS_PER_CELL = 1000   # Bounded number of records per week to keep pilot fast (<2 mins)

# 5. OFFLINE DRY RUN:
DRY_RUN = False               # False = real API calls; True = synthetic offline test

print(f"Pilot configuration ready:")
print(f"  • Mode: {PILOT_MODE}")
print(f"  • Target(s): {PILOT_SUBS if PILOT_MODE == 'SUBREDDIT_LIST' else ['ALL_REDDIT']}")
print(f"  • Weeks to sample: {len(PILOT_WEEKS)}")

In [ ]:
# Bootstrap: locate the repo root and add it to sys.path so the shared 'src'
# package is importable regardless of the notebook's current working directory.
import os, sys
from pathlib import Path
def _has_root_marker(p):
    try:
        return p.is_dir() and (p / "config/project_config.yaml").is_file()
    except OSError:
        return False
def _find_root(start):
    for p in [start, *start.parents]:
        if _has_root_marker(p):
            return p
    for depth in (1, 2):
        for sub in start.glob("/".join(["*"] * depth)):
            if sub.is_dir() and _has_root_marker(sub):
                return sub
    return None
_repo_root = _find_root(Path.cwd().resolve())
if _repo_root is not None and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
# Cell 2 — Setup & Helpers
import os, sys, csv, json, gzip, hashlib, datetime, time
from pathlib import Path
import yaml

from src.paths import get_project_root, resolve_tmp
from src.storage import atomic_write_text, sha256_file
from src.manifests import RETRIEVAL_COLS, load_manifest, upsert_manifest_row
from src.api import api_get, retry_get
from src.cleaner import clean_and_tokenize, extract_text, lang_of, _tests

ROOT = get_project_root()
CFG_PATH = ROOT / "config/project_config.yaml"
cfg = yaml.safe_load(open(CFG_PATH, encoding="utf-8"))
CFG_SHA = sha256_file(CFG_PATH)
TMP = resolve_tmp(ROOT, cfg)

RMAN = ROOT / "manifests/pilot/retrieval_manifest.csv"
if not RMAN.exists():
    atomic_write_text(RMAN, ",".join(RETRIEVAL_COLS) + "\n")

def rrows(): return load_manifest(RMAN)
manifest_rows = rrows
def manifest_upsert(row): upsert_manifest_row(RMAN, row, ["unit_id"], RETRIEVAL_COLS)

def dt_to_unix(d_str):
    return int(datetime.datetime.fromisoformat(d_str).replace(tzinfo=datetime.timezone.utc).timestamp())

print(f"Setup complete | Config version: {cfg['config_version']} | Scratch dir: {TMP}")

In [ ]:
# Cell 3 — Source Liveness Probes (Checking Arctic Shift API Availability)
print("Probing Arctic Shift API endpoints...")

probes = {}
if DRY_RUN:
    probes["status"] = "DRY_RUN (Network skipped)"
    print("  [DRY_RUN] Skipping live network probes.")
else:
    # Test Posts search endpoint
    try:
        r = retry_get("https://arctic-shift.photon-reddit.com/api/posts/search",
                      params={"subreddit": PILOT_SUBS[0] if PILOT_MODE == "SUBREDDIT_LIST" else "askscience", "limit": 2}, tries=3)
        data = r.json().get("data", [])
        probes["arctic_posts_api"] = f"ONLINE (Retrieved {len(data)} sample posts)"
        print(f"  [OK] Posts Search API: ONLINE")
    except Exception as e:
        probes["arctic_posts_api"] = f"FAILED: {str(e)[:100]}"
        print(f"  [NOTE] Posts Search API: {str(e)[:100]}")

    # Test Comments search endpoint
    try:
        r = retry_get("https://arctic-shift.photon-reddit.com/api/comments/search",
                      params={"subreddit": PILOT_SUBS[0] if PILOT_MODE == "SUBREDDIT_LIST" else "askscience", "limit": 2}, tries=3)
        data = r.json().get("data", [])
        probes["arctic_comments_api"] = f"ONLINE (Retrieved {len(data)} sample comments)"
        print(f"  [OK] Comments Search API: ONLINE")
    except Exception as e:
        probes["arctic_comments_api"] = f"FAILED: {str(e)[:100]}"
        print(f"  [NOTE] Comments Search API: {str(e)[:100]}")

probe_file = ROOT / "diagnostics/retrieval/source_probes.json"
probe_file.parent.mkdir(parents=True, exist_ok=True)
atomic_write_text(probe_file, json.dumps({"timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(), "probes": probes}, indent=2))
print(f"Source probe results saved to: {probe_file.name}")

In [ ]:
# Cell 4 — Text Cleaner Demonstration
# Shows how raw Reddit markdown is cleaned into normalized tokens for Word2Vec

SAMPLE_TEXTS = [
    "I am NOT happy with the tenure rules on r/AskAcademia!",
    "Check out this study https://doi.org/10.1000/182 and ask u/researcher_jane.",
    "PhD thesis defense is never easy 🎓🔬. Code example: ```python print('done')```",
    "Values were 1024 and 100000 participants in the 2019 survey.",
    "[deleted]"
]

print("=" * 80)
print("TEXT CLEANING & TOKENIZATION PREVIEW:")
print("=" * 80)

for idx, raw in enumerate(SAMPLE_TEXTS, 1):
    tokens, flags = clean_and_tokenize(raw)
    print(f"Example {idx}:")
    print(f"  Raw input:  {raw}")
    print(f"  Clean toks: {tokens}")
    print(f"  Status:     {flags['dropped'] if flags['dropped'] else 'KEPT (Valid)'}")
    print("-" * 80)

In [ ]:
# Cell 5 — Retrieve Pilot Slices & Save Shards
target_subs = PILOT_SUBS if PILOT_MODE == "SUBREDDIT_LIST" else ["ALL_REDDIT"]
types = ["comments", "submissions"]
endpoint_map = {
    "comments": "https://arctic-shift.photon-reddit.com/api/comments/search",
    "submissions": "https://arctic-shift.photon-reddit.com/api/posts/search"
}

inspection_rows = []
today_str = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
n_done, n_skip, n_fail = 0, 0, 0

print(f"Retrieving pilot sample ({len(target_subs)} targets x {len(PILOT_WEEKS)} weeks x 2 content types)...")

for sub in target_subs:
    for ws, we in PILOT_WEEKS:
        for ctype in types:
            unit_id = f"pilot__{sub}__{ctype}__{ws}_{we}"
            prior = {r["unit_id"]: r for r in manifest_rows()}.get(unit_id)
            
            # Skip if already completed
            if prior and prior.get("status") == "complete":
                out_f = Path(prior.get("output_final", ""))
                if out_f.exists() and sha256_file(out_f) == prior.get("output_sha256", ""):
                    n_skip += 1
                    continue
            
            out_path = ROOT / f"shards/tokenized/pilot/pilot__{sub}__{ws}__{ctype}__0000.jsonl.gz"
            out_path.parent.mkdir(parents=True, exist_ok=True)
            
            try:
                after, end_u = dt_to_unix(ws), dt_to_unix(we)
                if DRY_RUN:
                    recs = [
                        {"id": f"{ctype[:1]}_{i}", "created_utc": after + i,
                         "body": f"Pilot record {i} about research and mentoring in {sub}, not difficult but thorough.",
                         "title": f"Post title {i}", "selftext": "Body text here."}
                        for i in range(150)
                    ]
                else:
                    recs = []
                    params = {"after": after, "before": end_u, "limit": 100, "sort": "asc"}
                    if sub != "ALL_REDDIT":
                        params["subreddit"] = sub
                    while len(recs) < MAX_RECORDS_PER_CELL:
                        r = retry_get(endpoint_map[ctype], params=params, tries=3)
                        batch = r.json().get("data", [])
                        if not batch: break
                        recs.extend(batch)
                        after = int(batch[-1].get("created_utc", after)) + 1
                        params["after"] = after
                        if len(batch) < 100: break
                
                tmp_out = out_path.with_suffix(out_path.suffix + ".tmp")
                n_read = n_use = n_tok = 0
                
                with gzip.open(tmp_out, "wt", encoding="utf-8") as fz:
                    fz.write("#manifest " + json.dumps({"unit": unit_id, "config": cfg["config_version"]}) + "\n")
                    for rec in recs:
                        n_read += 1
                        raw_text = extract_text(ctype, rec)
                        tokens, fl = clean_and_tokenize(raw_text)
                        if len(inspection_rows) < 50:
                            inspection_rows.append({"sub": sub, "type": ctype, "raw": raw_text[:120], "tokens": " ".join(tokens[:20]), "kept": bool(tokens)})
                        if not tokens:
                            continue
                        rid = rec.get("id", f"id_{n_read}")
                        rec_out = {"rid_hash": hashlib.sha256(str(rid).encode()).hexdigest()[:16],
                                   "sub": sub, "ts": datetime.datetime.fromtimestamp(int(rec.get("created_utc", after)), datetime.timezone.utc).isoformat(),
                                   "ctype": ctype[:3], "tokens": tokens}
                        fz.write(json.dumps(rec_out) + "\n")
                        n_use += 1; n_tok += len(tokens)
                
                os.replace(tmp_out, out_path)
                sha_val = sha256_file(out_path)
                manifest_upsert({
                    "unit_id": unit_id, "source": "arctic_shift_api", "subreddit": sub,
                    "start_ts": ws + "T00:00:00Z", "end_ts": we + "T00:00:00Z", "content_type": ctype,
                    "status": "complete", "attempt_count": 1,
                    "started_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                    "completed_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                    "n_read": n_read, "n_usable": n_use, "token_estimate": n_tok,
                    "output_final": str(out_path), "output_sha256": sha_val,
                    "config_version": cfg["config_version"], "config_sha256": CFG_SHA,
                    "retrieval_date": today_str
                })
                n_done += 1
                print(f"  [OK] {unit_id}: read={n_read}, usable={n_use}, tokens={n_tok}")
            except Exception as e:
                n_fail += 1
                print(f"  [FAIL] {unit_id}: {e}")

print(f"\nPilot retrieval finished: {n_done} completed, {n_skip} skipped, {n_fail} failed.")

In [ ]:
# Cell 6 — Visual Inspection Table (Data Quality Check)
import pandas as pd

if inspection_rows:
    df = pd.DataFrame(inspection_rows)
    print("=" * 80)
    print("DATA QUALITY INSPECTION SAMPLE (First 10 records):")
    print("=" * 80)
    display(df.head(10))
    
    # Save sample to diagnostics
    insp_csv = ROOT / "diagnostics/retrieval/pilot_inspection_sample.csv"
    df.to_csv(insp_csv, index=False)
    print(f"Inspection sample saved to: {insp_csv.name}")
else:
    print("No inspection rows collected.")

In [ ]:
# Cell 7 — END-OF-PILOT SUMMARY
print("=" * 70)
print("STAGE 1 PILOT COMPLETE!")
print(f"  • Mode tested: {PILOT_MODE}")
print(f"  • Cells completed: {n_done} (skipped: {n_skip})")
print(f"  • Manifest updated: manifests/pilot/retrieval_manifest.csv")
print("\nDecision: If the cleaned text in the table above looks good,")
print("proceed to '02_counts_and_periods.ipynb' to calculate monthly volumes and freeze periods.")
print("=" * 70)